# 00a. Vectorization

시뮬레이션 데이터 → Descriptor Vector 변환 파이프라인.

**4가지 벡터화 방법:**
1. **Ordinary PI** — VR filtration → PH → Persistence Image
2. **Six-pack (Rips)** — L⊆K inclusion의 6가지 barcode → PI
3. **Six-pack (Chromatic)** — Chromatic Alpha Complex 기반
4. **Mixup Barcode** — Interaction PI & 3D PI

**필요 패키지:** `gudhi`, `persim`, `chromatic_tda`, `scipy`

In [ ]:
# !pip install gudhi persim chromatic-tda
import os
from gudhi import RipsComplex
import numpy as np
import matplotlib.pyplot as plt
from persim import PersistenceImager
import persim.images_weights as weights
from collections import defaultdict
from typing import Dict, List
from scipy.stats import norm

---
## 1. Ordinary Persistence Image (Ord_PI)

각 sub-population(Red/Green/All)에 독립 VR filtration → PH → PI 수행.
- H0: birth_range=(0,1), pers_range=(0,10), mean(axis=0) → 100D
- H1: birth_range=(0,10), pers_range=(0,5), flatten → 5000D

In [ ]:
def compute_Rips(points, max_edge=10):
    rips = RipsComplex(points=points, max_edge_length=max_edge)
    return rips.create_simplex_tree(max_dimension=2)

def compute_Persistence_barcode(A):
    fil_A = compute_Rips(A)
    fil_A.persistence()
    bar_A = {}
    for dim in [0, 1]:
        bars = fil_A.persistence_intervals_in_dimension(dim)
        bar_A[dim] = np.array([[b,d] for b,d in bars if d != np.inf and d-b > 1e-5])
        if len(bar_A[dim]) == 0: bar_A[dim] = np.zeros((0,2))
    return bar_A

def compute_PIs(barcodes, max_eps=10, px_res=0.1, sigma=0.05, normalization=False):
    """Persistence barcode → PI vector (H0: 100D, H1: 5000D)."""
    for key in barcodes:
        if len(barcodes[key]) == 0: barcodes[key] = np.zeros((0, 2))
    vector = {}
    # H0
    pim_h0 = PersistenceImager()
    pim_h0.pixel_size = px_res
    pim_h0.birth_range = (0, 1); pim_h0.pers_range = (0, max_eps)
    pim_h0.weight = weights.persistence; pim_h0.weight_params = {'n': 1}
    pim_h0.kernel_params = {'sigma': [[sigma,0],[0,sigma]]}
    bars_h0 = np.array(barcodes[0])
    if len(bars_h0) > 0:
        img_h0 = pim_h0.transform(bars_h0, skew=False)
    else:
        img_h0 = np.zeros((int(1/px_res), int(max_eps/px_res)))
    vector[0] = np.mean(img_h0, axis=0)
    # H1
    pim_h1 = PersistenceImager()
    pim_h1.pixel_size = px_res
    pim_h1.birth_range = (0, max_eps); pim_h1.pers_range = (0, max_eps/2)
    pim_h1.weight = weights.persistence; pim_h1.weight_params = {'n': 1}
    pim_h1.kernel_params = {'sigma': [[sigma,0],[0,sigma]]}
    bars_h1 = np.array(barcodes[1])
    if len(bars_h1) > 0:
        img_h1 = pim_h1.transform(bars_h1, skew=True)
    else:
        img_h1 = np.zeros((int(max_eps/px_res), int((max_eps/2)/px_res)))
    if normalization:
        if np.max(vector[0])>0: vector[0] = vector[0]/np.max(vector[0])
        if np.max(img_h1)>0: vector[1] = img_h1.flatten()/np.max(img_h1)
        else: vector[1] = img_h1.flatten()
    else:
        vector[1] = img_h1.flatten()
    return vector

def visualize_PIs(PIs, max_eps=10, px_res=0.1):
    h0_img = PIs[0]
    h1_img = PIs[1].reshape((int(max_eps/px_res), int((max_eps/2)/px_res)))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h0_img, 'b-', linewidth=1.5)
    axes[0].fill_between(range(len(h0_img)), h0_img, alpha=0.3)
    axes[0].set_title('H0 PI (1D)'); axes[0].grid(True, alpha=0.3)
    im = axes[1].imshow(h1_img.T, cmap='hot', origin='lower', aspect='auto')
    axes[1].set_title('H1 PI (2D)'); plt.colorbar(im, ax=axes[1])
    plt.tight_layout(); plt.show()

In [ ]:
# === Ordinary PI 실행 예시 ===
A = np.random.rand(120, 2)*20
B = np.random.rand(80, 2)*20

PB_total = compute_Persistence_barcode(np.concatenate([A, B]))
PB_A = compute_Persistence_barcode(A)
PB_B = compute_Persistence_barcode(B)

PI_total = compute_PIs(PB_total)
PI_A = compute_PIs(PB_A)
PI_B = compute_PIs(PB_B)
visualize_PIs(PI_B)

---
## 2. Six-pack (Rips)

L=Rips(A) ⊆ K=Rips(A∪B) inclusion에서 6가지 barcode 계산:
- **Image, Kernel, Cokernel** (boundary matrix reduction)
- **Complex, Sub_complex, Relative** (ordinary PH)
- 양방향: A→B, B→A

In [ ]:
def divide_filtration(st):
    pairs = [(tuple(sorted(s)), f) for s, f in st.get_filtration()]
    return [p[0] for p in pairs], [p[1] for p in pairs]

def _build_boundary(simplices):
    sf_to_idx = {s: i for i, s in enumerate(simplices)}
    boundary = []
    for s in simplices:
        if len(s) <= 1: boundary.append(set())
        else:
            rows = set()
            for j in range(len(s)):
                face = s[:j] + s[j+1:]
                if face in sf_to_idx: rows.add(sf_to_idx[face])
            boundary.append(rows)
    return boundary

def _reduce_with_V(columns):
    m = len(columns)
    R = [set(col) for col in columns]
    V = [{i} for i in range(m)]
    low = [-1] * m; pivot_of_row = {}
    for i in range(m):
        while R[i]:
            li = max(R[i])
            if li in pivot_of_row:
                owner = pivot_of_row[li]
                R[i] ^= R[owner]; V[i] ^= V[owner]
            else:
                pivot_of_row[li] = i; low[i] = li; break
        else: low[i] = -1
    return R, low, V

def compute_all_barcodes(A, B, max_edge=10):
    """Compute Image, Kernel, Cokernel barcodes for L ⊆ K."""
    total = np.concatenate([A, B]); a = len(A)
    st = compute_Rips(total, max_edge=5)
    simplices, filt = divide_filtration(st); m = len(simplices)
    in_L = [all(v < a for v in s) for s in simplices]
    idx_L = [i for i, b in enumerate(in_L) if b]
    idx_KmL = [i for i, b in enumerate(in_L) if not b]
    g2L = {g: pos for pos, g in enumerate(idx_L)}
    Df = _build_boundary(simplices)
    Rf, lowf, Vf = _reduce_with_V(Df)
    boundary_L = [{g2L[r] for r in Df[g] if r in g2L} for g in idx_L]
    Rg, lowg, Vg = _reduce_with_V(boundary_L)
    row_order = idx_L + idx_KmL
    row_remap = {g: i for i, g in enumerate(row_order)}
    Dim = [{row_remap[r] for r in Df[c]} for c in range(m)]
    Rim, lowim, _ = _reduce_with_V(Dim)
    Vim = [{row_remap[r] for r in Vf[c]} for c in range(m)]
    cycle_cols = [i for i in range(m) if not Rim[i]]
    Dker = [Vim[c] for c in cycle_cols]
    _, lowker, _ = _reduce_with_V(Dker) if Dker else ([], [], [])
    cycle_pos = {c: pos for pos, c in enumerate(cycle_cols)}
    Dcok = []
    for k_idx in idx_KmL:
        col = {row_remap[r] for r in Df[k_idx]}
        Dcok.append(col)
    _, lowcok, _ = _reduce_with_V(Dcok) if Dcok else ([], [], [])
    # Extract barcodes
    barcodes = {'image': {0:[], 1:[]}, 'kernel': {0:[], 1:[]}, 'cokernel': {0:[], 1:[]}}
    for i in range(m):
        if lowim[i] >= 0:
            dim_s = len(simplices[i]) - 1
            if dim_s > 0:
                birth = filt[lowim[i]] if lowim[i] < len(filt) else 0
                death = filt[i]
                if abs(death - birth) > 1e-10:
                    barcodes['image'][dim_s-1].append([birth, death])
    for pos, c in enumerate(cycle_cols):
        if lowker and pos < len(lowker) and lowker[pos] >= 0:
            dim_s = len(simplices[c]) - 1
            birth_idx = cycle_cols[lowker[pos]] if lowker[pos] < len(cycle_cols) else 0
            birth = filt[birth_idx] if birth_idx < len(filt) else 0
            death = filt[c]
            if abs(death - birth) > 1e-10:
                barcodes['kernel'][dim_s].append([birth, death])
    for pos, k_idx in enumerate(idx_KmL):
        if lowcok and pos < len(lowcok) and lowcok[pos] >= 0:
            dim_s = len(simplices[k_idx]) - 1
            birth_orig = idx_KmL[lowcok[pos]] if lowcok[pos] < len(idx_KmL) else 0
            birth = filt[birth_orig] if birth_orig < len(filt) else 0
            death = filt[k_idx]
            if abs(death - birth) > 1e-10:
                barcodes['cokernel'][dim_s-1].append([birth, death])
    for k in barcodes:
        for d in [0,1]:
            barcodes[k][d] = np.array(barcodes[k][d]) if barcodes[k][d] else np.zeros((0,2))
    return barcodes

In [ ]:
# === Six-pack (Rips) 실행 예시 ===
six_pack_A_to_B = compute_all_barcodes(A, B)
six_pack_B_to_A = compute_all_barcodes(B, A)

# Ordinary PH 추가 (complete six-pack)
PB_total = compute_Persistence_barcode(np.concatenate([A, B]))
PB_A = compute_Persistence_barcode(A)
PB_B = compute_Persistence_barcode(B)
six_pack_A_to_B.update({'complex': PB_total, 'sub_complex': PB_A, 'relative': PB_B})
six_pack_B_to_A.update({'complex': PB_total, 'sub_complex': PB_B, 'relative': PB_A})

# PI 벡터화
PI_six_A = {k: compute_PIs(six_pack_A_to_B[k]) for k in six_pack_A_to_B}
PI_six_B = {k: compute_PIs(six_pack_B_to_A[k]) for k in six_pack_B_to_A}
print('Six-pack keys:', list(PI_six_A.keys()))

---
## 3. Six-pack (Chromatic Alpha)

Chromatic Alpha Complex 기반 six-pack. `chromatic_tda` 라이브러리 사용.

In [ ]:
# chromatic_tda가 설치된 경우에만 실행
try:
    import chromatic_tda as chro

    def convert_into_diagram(diagram):
        diagrams = {}
        for dim, bars in diagram.items():
            cleaned = [[b,d] for b,d in bars if d != np.inf and d-b > 1e-5]
            diagrams[dim] = np.array(cleaned) if cleaned else np.zeros((0,2))
        return diagrams

    def compute_six_pack_chroma(points, labels, max_edge=10):
        chro_alpha = chro.ChromaticAlphaComplex(points, labels, max_alpha=max_edge)
        st = chro_alpha.get_simplicial_complex(sub_complex='0', full_complex='all')
        six_pack = st.bars_six_pack()
        result = {}
        for key in ['image', 'kernel', 'cokernel', 'domain', 'codomain', 'relative']:
            if key in six_pack:
                result[key] = convert_into_diagram(six_pack[key])
        return result

    # === 실행 예시 ===
    points = np.concatenate([A, B])
    labels = np.concatenate([np.zeros(len(A)), np.ones(len(B))])
    sp_chroma_A = compute_six_pack_chroma(points, labels)
    PI_chroma = {k: compute_PIs(sp_chroma_A[k]) for k in sp_chroma_A}
    print('Chromatic six-pack keys:', list(PI_chroma.keys()))

except ImportError:
    print('chromatic_tda not installed. Skipping.')

---
## 4. Mixup Barcode → Interaction PI & 3D PI

Wagner et al. (2024) 기반 Canonical Matching:
- **Interaction PI**: (b, d) 좌표에 weight=mixup (d-d') 적용
- **3D PI**: (b, d'-b, d-b) 좌표의 3D Gaussian KDE

In [ ]:
def extract_filtration(st):
    pairs = [(tuple(sorted(s)), f) for s, f in st.get_filtration()]
    return [p[0] for p in pairs], [p[1] for p in pairs]

def compute_mixup_barcode(A, B, max_edge=10, max_dim=1):
    """Compute Mixup Barcode via canonical matching."""
    total = np.concatenate([A, B]); a = len(A)
    st_K = compute_Rips(total, max_edge=max_edge)
    simplicesK, filtK = extract_filtration(st_K)
    mK = len(simplicesK)
    # Identify L simplices
    in_L = [all(v < a for v in s) for s in simplicesK]
    # Build & reduce K boundary
    DK = _build_boundary(simplicesK)
    RK, lowK, VK = _reduce_with_V(DK)
    pivotK_to_col = {lowK[i]: i for i in range(mK) if lowK[i] >= 0}
    # Build & reduce L boundary (reindexed)
    idx_L = [i for i, b in enumerate(in_L) if b]
    g2L = {g: pos for pos, g in enumerate(idx_L)}
    DL = [{g2L[r] for r in DK[g] if r in g2L} for g in idx_L]
    RL, lowL, VL = _reduce_with_V(DL)
    pivotL_to_col = {lowL[i]: i for i in range(len(idx_L)) if lowL[i] >= 0}
    # Extract mixup triples
    mixup_triples = defaultdict(list)
    for pos_j, g_j in enumerate(idx_L):
        if lowL[pos_j] >= 0:  # j is a destroyer in L
            dim = len(simplicesK[g_j]) - 2
            sigma_row = lowL[pos_j]  # pivot row in L
            sigma_global = idx_L[sigma_row]
            birth = filtK[sigma_global]
            death = filtK[g_j]
            # Find death in K
            tau_prime = pivotK_to_col.get(sigma_global, None)
            death_prime = filtK[tau_prime] if tau_prime is not None else np.inf
            if not np.isinf(death) and (np.isinf(death_prime) or death_prime > death):
                death_prime = death
            if np.isinf(death) or abs(death - birth) > 1e-10:
                mixup_triples[dim].append((birth, death_prime, death))
    result = {}
    for dim in sorted(mixup_triples.keys()):
        arr = np.array(mixup_triples[dim])
        result[dim] = arr[np.argsort(arr[:, 0])]
    for d in range(max_dim + 1):
        if d not in result: result[d] = np.empty((0, 3))
    return result

In [ ]:
def compute_Interaction_PIs(barcodes, max_eps=10, px_res=0.1, sigma=0.05,
                            weight_type='mixup'):
    """Mixup barcode → Interaction PI. weight: 'mixup' | 'mixup_persistence'"""
    vector = {}
    def make_weight(b_arr, d_prime_arr, d_arr):
        if weight_type == 'mixup': w_arr = d_arr - d_prime_arr
        else: w_arr = 2*d_arr - d_prime_arr - b_arr
        counter = {'i': 0}
        def weight_fn(birth, persistence, **kw):
            val = float(w_arr[counter['i']])
            counter['i'] = (counter['i']+1) % len(w_arr)
            return val
        return weight_fn
    for dim, br, pr in [(0, (0,0.01), (0,max_eps)), (1, (0,max_eps), (0,max_eps/2))]:
        pim = PersistenceImager()
        pim.pixel_size = px_res; pim.birth_range = br; pim.pers_range = pr
        pim.kernel_params = {'sigma': [[sigma,0],[0,sigma]]}
        bars = np.asarray(barcodes.get(dim, np.zeros((0,3))))
        if len(bars) > 0:
            mask = np.all(np.isfinite(bars), axis=1)
            b, dp, d = bars[mask,0], bars[mask,1], bars[mask,2]
            if len(b) > 0:
                pim.weight = make_weight(b, dp, d)
                img = pim.transform(np.stack([b,d],axis=1), skew=True)
            else: img = np.zeros((int(br[1]/px_res) if br[1]>0.1 else 1, int(pr[1]/px_res)))
        else: img = np.zeros((int(br[1]/px_res) if br[1]>0.1 else 1, int(pr[1]/px_res)))
        vector[dim] = np.mean(img, axis=0) if dim==0 else img.flatten()
    return vector

def compute_3d_PI(mixup_barcodes, resolution=20, ranges=((0,10),(0,10),(0,10)),
                  bandwidth=0.1, weight_type='ones'):
    """Mixup barcode → 3D Persistence Image."""
    def translate(mb):
        out = {}
        for i in range(2):
            out[i] = [[b, dp-b, d-b] for b,dp,d in mb.get(i,[])
                      if np.isfinite(b) and np.isfinite(dp) and np.isfinite(d)]
        return out
    wfn = {'mixup': lambda p: p[2]-p[1],
           'mixup_persistence': lambda p: 2*p[2]-p[1],
           'ones': lambda p: 1.0}[weight_type]
    translated = translate(mixup_barcodes)
    vectors = {}
    grids = [np.linspace(r[0],r[1],resolution) for r in ranges]
    for i in range(2):
        if i == 0:
            img = np.zeros((resolution, resolution))
            for p in translated[i]:
                if all(np.isfinite(v) for v in p):
                    img += wfn(p) * np.outer(norm.pdf(grids[1],p[1],bandwidth),
                                             norm.pdf(grids[2],p[2],bandwidth))
        else:
            img = np.zeros((resolution, resolution, resolution))
            for p in translated[i]:
                if all(np.isfinite(v) for v in p):
                    img += wfn(p) * np.einsum('i,j,k->ijk',
                        norm.pdf(grids[0],p[0],bandwidth),
                        norm.pdf(grids[1],p[1],bandwidth),
                        norm.pdf(grids[2],p[2],bandwidth))
        vectors[i] = img.flatten()
    return vectors

In [ ]:
# === Mixup Barcode 실행 예시 ===
mixup_A = compute_mixup_barcode(A, B)
mixup_B = compute_mixup_barcode(B, A)

# Interaction PI
inter_PI_A = compute_Interaction_PIs(mixup_A, weight_type='mixup')
inter_PI_B = compute_Interaction_PIs(mixup_B, weight_type='mixup')
print(f'Inter_PI dims: H0={inter_PI_A[0].shape}, H1={inter_PI_A[1].shape}')
visualize_PIs(inter_PI_A)

# 3D PI
pi3d_A = compute_3d_PI(mixup_A, weight_type='ones')
pi3d_B = compute_3d_PI(mixup_B, weight_type='ones')
print(f'3D_PI dims: H0={pi3d_A[0].shape}, H1={pi3d_A[1].shape}')